In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings('ignore')

# ====================== PATHS ======================
CLEAN_CSV = '/content/drive/MyDrive/Sample/Training_Data_Clean.csv'
MODEL_PATH = '/content/drive/MyDrive/Sample/book_replacement_model_final.pkl'

print("Loading data...\n")

df = pd.read_csv(CLEAN_CSV)

# ====================== FEATURES (Removed value for money to break leakage) ======================
feature_cols = [
    "How easy was this textbook to understand?_score",
    "How would you rate the conceptual clarity of the book?_score",
    "How engaging and interactive were the examples and exercises?_score",
    "How helpful were the visuals, diagrams, and illustrations?_score",
    "Overall, how satisfied are you with this textbook?_score",
    "Would you recommend this book to other students?_score",
    # "How would you rate the value for money..."  ← Removed to reduce leakage
    "Was the book affordable given your financial situation?_score",
    "How useful was this book for scoring well in exams?_score",
    "How likely are you to use this book again in future courses?_score"
]

X = df[feature_cols]
y = df['Needs_Replacement']

print(f"Total samples: {len(df):,}")
print(f"Target distribution:\n{y.value_counts(normalize=True).round(4)*100}")

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

# Train Model with strong regularization
model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.08,
    subsample=0.75,
    colsample_bytree=0.75,
    reg_lambda=1.5,      # L2 regularization
    reg_alpha=0.5,       # L1 regularization
    gamma=0.5,           # Minimum loss reduction
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=42,
    eval_metric='auc'
)

model.fit(X_train, y_train)

# Evaluation
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("\n" + "="*65)
print("FINAL MODEL EVALUATION (After Leakage Fix)")
print("="*65)
print(f"Accuracy     : {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC      : {roc_auc_score(y_test, y_pred_proba):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Keep (0)', 'Replace (1)']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Feature Importance
importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 8 Important Features:")
print(importance.head(8))

# Save Model
joblib.dump(model, MODEL_PATH)
print(f"\n✅ Model saved at: {MODEL_PATH}")

# ====================== PREDICTION FUNCTION ======================
def predict_new_form(feedback_dict):
    input_data = {}
    for base_q in [
        "How easy was this textbook to understand?",
        "How would you rate the conceptual clarity of the book?",
        "How engaging and interactive were the examples and exercises?",
        "How helpful were the visuals, diagrams, and illustrations?",
        "Overall, how satisfied are you with this textbook?",
        "Would you recommend this book to other students?",
        "Was the book affordable given your financial situation?",
        "How useful was this book for scoring well in exams?",
        "How likely are you to use this book again in future courses?"
    ]:
        ans = feedback_dict.get(base_q, "Average")
        input_data[base_q + "_score"] = score_map.get(ans, 3)

    input_df = pd.DataFrame([input_data])

    loaded_model = joblib.load(MODEL_PATH)
    pred = loaded_model.predict(input_df)[0]
    prob = loaded_model.predict_proba(input_df)[0][1]

    result = "🔍 Search for Alternative (Replace)" if pred == 1 else "✅ Keep this book"

    print("\n" + "="*70)
    print("PREDICTION ON NEW FORM DATA")
    print("="*70)
    print(f"Prediction     : {result}")
    print(f"Confidence     : {prob:.1%}")
    print("="*70)

    return pred, prob

# ====================== COUPLE OF EXAMPLES ======================
print("\n=== Example Predictions ===")

ex1 = {
    "How easy was this textbook to understand?": "Good",
    "How would you rate the conceptual clarity of the book?": "Good",
    "How engaging and interactive were the examples and exercises?": "Good",
    "How helpful were the visuals, diagrams, and illustrations?": "Good",
    "Overall, how satisfied are you with this textbook?": "Good",
    "Would you recommend this book to other students?": "Good",
    "Was the book affordable given your financial situation?": "Average",
    "How useful was this book for scoring well in exams?": "Good",
    "How likely are you to use this book again in future courses?": "Good"
}
predict_new_form(ex1)

ex2 = {
    "How easy was this textbook to understand?": "Poor",
    "How would you rate the conceptual clarity of the book?": "Poor",
    "How engaging and interactive were the examples and exercises?": "Poor",
    "How helpful were the visuals, diagrams, and illustrations?": "Average",
    "Overall, how satisfied are you with this textbook?": "Poor",
    "Would you recommend this book to other students?": "Poor",
    "Was the book affordable given your financial situation?": "Poor",
    "How useful was this book for scoring well in exams?": "Average",
    "How likely are you to use this book again in future courses?": "Poor"
}
predict_new_form(ex2)

Loading data...

Total samples: 912,348
Target distribution:
Needs_Replacement
0    79.21
1    20.79
Name: proportion, dtype: float64
Train: 729,878 | Test: 182,470

FINAL MODEL EVALUATION (After Leakage Fix)
Accuracy     : 0.7429
ROC-AUC      : 0.8524

Classification Report:
              precision    recall  f1-score   support

    Keep (0)       0.99      0.68      0.81    144541
 Replace (1)       0.45      0.97      0.61     37929

    accuracy                           0.74    182470
   macro avg       0.72      0.83      0.71    182470
weighted avg       0.87      0.74      0.77    182470


Confusion Matrix:
[[98862 45679]
 [ 1240 36689]]

Top 8 Important Features:
                                             Feature  Importance
0    How easy was this textbook to understand?_score    0.734478
6  Was the book affordable given your financial s...    0.039676
1  How would you rate the conceptual clarity of t...    0.034385
7  How useful was this book for scoring well in e...    0.0

(np.int64(1), np.float32(0.8433108))